# Input-embedding vs. per-iteration-output similarity

For each recurrence iteration `r`, measure the cosine similarity between
**the hidden state fed into the recurrent block** (the "first input embedding")
and **the raw output of iteration `r`** (`each_recurrence_hidden_states[r]`).

* Tokens: `.ppl_cache/openwebtext_n5000000_seed42.npy` (flat uint32 GPT-2 token stream).
* Model dynamics: `src/modalities/models/gpt2/gpt2_model.py` (`GroupRecursiveGPT2MTPBlock`).
* Metric: per-token cosine (`dim=-1`), averaged over sequence positions and all windows.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

%matplotlib inline

WORKSPACE_ROOT = Path('/raid/s3/opengptx/behzad_shomali')
MODALITIES_SRC = WORKSPACE_ROOT / 'modalities' / 'src'
if str(MODALITIES_SRC) not in sys.path:
    sys.path.insert(0, str(MODALITIES_SRC))

from modalities.evaluation.olmes_evaluator import load_modalities_model
from modalities.models.gpt2.gpt2_model import BlockTypes

## Configuration

**Fill in the checkpoint here.** `CONFIG_PATH = None` for a DCP checkpoint directory;
set it to the config `.yaml` for a non-DCP (single-file) checkpoint.

In [ ]:
# ----- Checkpoint (FILL THESE IN) -----
CHECKPOINT_PATH = ''           # <-- path to checkpoint dir (DCP) or file
CONFIG_PATH     = None         # <-- None for DCP; path to config yaml otherwise
MODEL_KEY       = 'model_raw'

# ----- Device / batching -----
DEVICE      = 'cuda:0' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE  = 4               # windows per forward pass
MAX_WINDOWS = None            # set to an int (e.g. 256) for a quick run; None = all
SEQ_LEN     = None            # None -> use model.sequence_length

# ----- Tokens -----
TOKEN_CACHE_FILE = Path(
    '/raid/s3/opengptx/behzad_shomali/modalities/loop_MTP_paper/.ppl_cache/openwebtext_n5000000_seed42.npy'
)

assert CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH), (
    f'Set CHECKPOINT_PATH to an existing checkpoint. Got: {CHECKPOINT_PATH!r}'
)
print('DEVICE =', DEVICE)

## Load model

We enable `return_each_recurrence_output` (on the model *and* the recurrent block)
so the forward pass exposes the per-iteration hidden states, and register a
forward-pre-hook on the recurrent block to capture its input tensor.

In [ ]:
temp_conversion_dir = tempfile.mkdtemp(prefix='modalities_converted_')
model, tokenizer, loaded_config = load_modalities_model(
    checkpoint_path=CHECKPOINT_PATH,
    config_path=CONFIG_PATH,
    model_key=MODEL_KEY,
    converted_output_dir=temp_conversion_dir,
)
model = model.to(DEVICE).eval()

# Expose per-iteration hidden states; skip per-iteration LM-head logits (not needed).
model.return_each_recurrence_output = True
model.need_mtp_logits = False
model.track_recurrence_embd_similarity = False

# Locate the recurrent block(s) and turn on their per-iteration output flag.
RECURRENT_TYPES = (BlockTypes.GROUP_RECURSIVE_MTP, BlockTypes.GROUP_RECURSIVE)
recurrent_layer_ids = [
    li for li in model.transformer.h
    if model.blocks_types[int(li)] in RECURRENT_TYPES
]
assert len(recurrent_layer_ids) >= 1, 'No recurrent (GROUP_RECURSIVE*) block found in this model.'
if len(recurrent_layer_ids) > 1:
    print(f'WARNING: {len(recurrent_layer_ids)} recurrent blocks found {recurrent_layer_ids}; '
          'hooking the FIRST one as the "input embedding" reference.')
REC_LAYER_ID = recurrent_layer_ids[0]
recurrent_block = model.transformer.h[REC_LAYER_ID]
recurrent_block.return_each_recurrence_output = True

K = int(recurrent_block.max_recurrence)
SEQ_LEN = SEQ_LEN or int(model.sequence_length)
print(f'Recurrent block at layer {REC_LAYER_ID}, max_recurrence (K) = {K}, seq_len = {SEQ_LEN}')

# Capture the tensor fed INTO the recurrent block (the "first input embedding").
_captured = {}
def _capture_block_input(module, args, kwargs):
    _captured['block_input'] = args[0].detach()
_hook_handle = recurrent_block.register_forward_pre_hook(_capture_block_input, with_kwargs=True)
print('Hook registered.')

## Window the token stream

In [ ]:
token_buffer = np.load(TOKEN_CACHE_FILE)
n_windows = len(token_buffer) // SEQ_LEN
windows = token_buffer[: n_windows * SEQ_LEN].reshape(n_windows, SEQ_LEN)
windows = torch.from_numpy(windows.astype(np.int64))
if MAX_WINDOWS is not None:
    windows = windows[:MAX_WINDOWS]
print(f'{windows.shape[0]:,} windows of length {SEQ_LEN}')

## Run the model and accumulate per-iteration similarity

For each window we compute, per iteration `r`,
`cos(block_input, each_recurrence_hidden_states[r])` per token position, then
accumulate the running mean over all token positions and windows.

In [ ]:
SAMPLE_KEY = model.sample_key  # key the model expects, e.g. 'input_ids'

cos_sum = torch.zeros(K, dtype=torch.float64)   # sum of per-token cosine over all positions
cos_sq_sum = torch.zeros(K, dtype=torch.float64)  # for std
token_count = 0

with torch.no_grad():
    for start in tqdm(range(0, windows.shape[0], BATCH_SIZE), desc='forward'):
        batch = windows[start:start + BATCH_SIZE].to(DEVICE)
        out = model({SAMPLE_KEY: batch})
        out = out[model.prediction_key] if model.prediction_key in out else out

        block_input = _captured['block_input'].float()              # (B, T, D)
        per_iter = out['each_recurrence_hidden_states']             # list of K x (B, T, D)
        assert len(per_iter) == K, (len(per_iter), K)

        for r, h_r in enumerate(per_iter):
            cos = F.cosine_similarity(block_input, h_r.float(), dim=-1)  # (B, T)
            cos = cos.reshape(-1).double().cpu()
            cos_sum[r] += cos.sum()
            cos_sq_sum[r] += (cos ** 2).sum()
        token_count += block_input.shape[0] * block_input.shape[1]

mean_cos = (cos_sum / token_count).numpy()
std_cos = np.sqrt((cos_sq_sum / token_count).numpy() - mean_cos ** 2)

results = pd.DataFrame({
    'iteration': np.arange(1, K + 1),
    'mean_cosine': mean_cos,
    'std_cosine': std_cos,
})
print(f'Aggregated over {token_count:,} token positions')
results

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(results['iteration'], results['mean_cosine'], yerr=results['std_cosine'],
            marker='o', capsize=3, linewidth=1.5)
ax.set_xlabel('Recurrence iteration')
ax.set_ylabel('Cosine similarity\n(block input vs. iteration output)')
ax.set_title('Input embedding vs. per-iteration output similarity\n(OpenWebText tokens)')
ax.set_xticks(results['iteration'])
ax.grid(True, linestyle=':', alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
Path('./plots').mkdir(exist_ok=True)
fig.savefig('./plots/input_vs_iteration_similarity.pdf')
plt.show()

In [ ]:
# Clean up the forward hook when done.
_hook_handle.remove()